In [2]:
import duckdb
import pandas as pd
import os
from datetime import datetime 

In [3]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [8]:
# ler arquivos csv com pandas
# Nome do arquivo que será utilizado
# Captura a data e hora atual (momento em que os dados estão sendo carregados)
# Lê o arquivo CSV que está dentro da pasta "landing"
# O separador ; é usado porque o arquivo provavelmente segue padrão brasileiro
# Cria uma nova coluna chamada "nome_arquivo"
# Essa coluna guarda o nome do arquivo de onde vieram os dados
# Cria uma nova coluna chamada "data_injestao"
# Essa coluna guarda o momento em que os dados foram importados
# Mostra as primeiras linhas do DataFrame para visualização
# quando foi realizando a injestao e qual arquivo que foi utilizando
# 
arquivo = 'z0019_2.csv'
data_ingestao = datetime.now()
df = pd.read_csv(f'../landing/{arquivo}',sep=';')
df['nome_arquivo'] = arquivo
df['data_ingestao'] = data_ingestao
df.head()
 

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,100,z0019_2.csv,2026-05-30 19:22:50.896163
1,10002,MARTELO,BT50,100,1500,z0019_2.csv,2026-05-30 19:22:50.896163
2,10003,PREGO,BT10,100,50,z0019_2.csv,2026-05-30 19:22:50.896163
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-05-30 19:22:50.896163
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-05-30 19:22:50.896163


In [5]:
con.execute("""
CREATE TABLE IF NOT EXISTS bronze_produtos(
        NATBR VARCHAR,
        MAKTX VARCHAR,
        WERKS VARCHAR,
        MAINS VARCHAR,
        LABST VARCHAR,
        nome_arquivo VARCHAR,
        data_ingestao TIMESTAMP) 
""")

In [13]:
con.execute("INSERT INTO bronze_z0019 SELECT * FROM df")

In [14]:
resultado = con.execute("SELECT * FROM bronze_z0019").fetchdf()
resultado.head(6)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-05-30 19:22:13.968230
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-05-30 19:22:13.968230
2,10003,PREGO,BT10,100,50,z0019_1.csv,2026-05-30 19:22:13.968230
3,10001,PARAFUSO,BT10,100,100,z0019_2.csv,2026-05-30 19:22:50.896163
4,10002,MARTELO,BT50,100,1500,z0019_2.csv,2026-05-30 19:22:50.896163
5,10003,PREGO,BT10,100,50,z0019_2.csv,2026-05-30 19:22:50.896163


In [11]:
con.execute("ALTER TABLE bronze_produtos RENAME TO bronze_z0019")

In [15]:
con.close()